# Module 3.1: Attention Mechanisms

We finally have all the ingredients: we've turned words into embeddings, and we've mathematically injected their position in the sentence. 

Now, we build the engine of the Transformer: **The Attention Mechanism**. This is where the model learns context by allowing every word to physically "look" at every other word.

## 1. The Core Idea: Learnable Q, K, V

### The Concept
In Module 1.1, we learned the math for $Attention(Q, K, V) = softmax(Q K^T / \sqrt{d_k})V$. 

But where do $Q$ (Query), $K$ (Key), and $V$ (Value) come from? We don't just use the raw embeddings. Instead, we use Neural Network **Linear Layers** (Weight Matrices) to mathematically *project* the embedding into three different versions of itself.

### Why do we need it? (Learnable Parameters)
If we just used the raw embeddings, the Attention formula would always output the exact same similarities (e.g. "Apple" would always attend to "Fruit"). By using `Linear` layers, we give the model "knobs" it can turn. During training, the gradients adjust these Linear layers so the model **learns** *what* it should be querying for depending on the task (e.g., in a translation task, it might learn to query for verbs).

## 2. Self-Attention (Single Head)

### The Concept
"Self-Attention" means the sequence is attending to *itself*. Every word in the sentence asks every other word: "Are you relevant to my meaning?"

For example, in the sentence "The bank of the river", the word `bank` will output a Query. The word `river` will have a Key that matches that Query well. They will have a high dot product, and `bank` will absorb the Value vector from `river`—nudging its own representation toward the "nature" sense rather than the "finance" sense.

> Note: with **random, untrained** weights this disambiguation does *not* happen — the projections are meaningless until trained. In the demo below we **hand-craft** the embeddings and projections so you can actually see `bank` attend to `river`.

### Why do we need it? (Parallel Context)
This replaces the old Recurrent Neural Networks (RNNs) which had to read left-to-right, one word at a time. Self-Attention evaluates the entire sentence synchronously using a single large Matrix Multiplication. This lets any word directly reference **any other word in the context window**—no "forgetting" of earlier words—and it runs fast on a GPU. The window is **finite**, though, and because every word compares against every other word, the compute and memory cost grows with the sequence length **squared** ($O(n^2)$). That quadratic cost is the main reason context windows can't simply be made infinite.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from math import sqrt

# Reproducibility: identical random weights on every run.
torch.manual_seed(0)

def scaled_dot_product_attention(query, key, value, mask=None):
    dk = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / sqrt(dk)
    if mask is not None:
        # Where the mask is 0 ("not allowed to look here"), set the score to a huge
        # negative number. After softmax, exp(-1e9) is effectively 0, so those
        # positions receive ~zero attention weight. -1e9 is just "large negative".
        scores = scores.masked_fill(mask == 0, -1e9)
    attention_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attention_weights, value)
    return output, attention_weights

class SingleHeadAttention(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        # These are the Learnable Weight Matrices!
        # bias=False: a Query/Key/Value projection only needs to ROTATE/SCALE the
        # embedding direction; a constant bias added to every token would not help
        # distinguish tokens and just wastes parameters. This matches real LLMs.
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        # x shape: (Batch, Seq_Len, d_model)
        Q = self.W_q(x)  # "What am I looking for?"
        K = self.W_k(x)  # "What do I have?"
        V = self.W_v(x)  # "What information will I give you?"

        # Use our math function from Module 1!
        # Output shape is the EXACT same as input shape (Batch, Seq_Len, d_model).
        # That matters later: because the shape is preserved, the attention output
        # can be ADDED back to the input as a residual connection (x + attention(x)).
        output, weights = scaled_dot_product_attention(Q, K, V, mask=mask)
        return output, weights

# 1 Batch, 5 Words, 128 Dimensional Embedding
dummy_sentence_embeddings = torch.randn(1, 5, 128)
single_head = SingleHeadAttention(d_model=128)

contextualized_output, _ = single_head(dummy_sentence_embeddings)
print(f"Input Shape : {dummy_sentence_embeddings.shape}")
print(f"Output Shape: {contextualized_output.shape}")
print("Same shape in and out -> can be used as a residual: x + attention(x)")

### Seeing it work: "bank" attends to "river"

The run above used random weights, so its attention pattern is meaningless. To actually *see* `bank` look at `river`, we **hand-craft** tiny 4-D embeddings where words that belong together point in similar directions, and we use identity Q/K/V projections (i.e. we pretend training already happened). Then we print the attention-weight matrix and a heatmap.

Sentence: `["the", "bank", "of", "the", "river"]`. We design "bank" and "river" to share a "nature" component so their dot product is high.

In [ ]:
import matplotlib.pyplot as plt

words = ["the", "bank", "of", "the", "river"]

# Hand-crafted 4-D embeddings. Dimensions (loosely):
# [function_word, finance, nature, water]
# "bank" and "river" both carry a strong "nature/water" component, so they match.
emb = torch.tensor([
    [1.0, 0.0, 0.0, 0.0],   # the   (function word)
    [0.0, 0.3, 0.9, 0.2],   # bank  (a bit of finance, lots of nature)
    [1.0, 0.0, 0.0, 0.0],   # of    (function word)
    [1.0, 0.0, 0.0, 0.0],   # the   (function word)
    [0.0, 0.0, 0.9, 0.9],   # river (nature + water)
]).unsqueeze(0)  # add batch dim -> (1, 5, 4)

# Use identity projections so Q = K = V = the embeddings themselves.
# (This is the "already trained, no extra transform" case, just to expose the mechanism.)
_, attn_weights = scaled_dot_product_attention(emb, emb, emb)

attn = attn_weights[0]  # (5, 5): row = "querying word", col = "word being looked at"

print("Attention weights (rows query, columns are attended-to):\n")
header = "          " + "".join(f"{w:>8}" for w in words)
print(header)
for i, w in enumerate(words):
    row = "".join(f"{attn[i, j]:8.2f}" for j in range(len(words)))
    print(f"{w:>10}{row}")

# The row for "bank" should put more weight on "river" than on the function words.
bank_i, river_i = words.index("bank"), words.index("river")
print(f"\n'bank' -> 'river' weight: {attn[bank_i, river_i]:.2f}  (highest among other words)")
print(f"'bank' -> 'the'   weight: {attn[bank_i, 0]:.2f}")

# Verify every row of a softmax is a valid probability distribution (sums to 1).
print("\nRow sums (should all be 1.0):", [round(v, 3) for v in attn.sum(dim=-1).tolist()])

plt.figure(figsize=(5, 4))
plt.imshow(attn.numpy(), cmap="viridis")
plt.xticks(range(len(words)), words)
plt.yticks(range(len(words)), words)
plt.xlabel("Looking AT")
plt.ylabel("Querying word")
plt.title("Attention weights")
plt.colorbar()
plt.show()

## 3. Multi-Head Attention (The Committee of Experts)

### The Analogy
Imagine reviewing a legal contract. If you review it alone, you might focus on the grammar but miss financial loopholes. 
Now imagine evaluating the contract with a **Committee of 8 Experts**. Expert 1 only checks grammar. Expert 2 only checks finances. Expert 3 tracks pronouns. 

**Multi-Head Attention (MHA)** is exactly this. Instead of one giant Attention layer, we run several smaller "Heads" in parallel and glue their findings back together at the end.

### A common misconception (important!)
We do **NOT** slice the raw embedding into 8 chunks and give each head a different slice of the original word vector. Instead:
1. One big `W_q` (and `W_k`, `W_v`) projects the **whole** `d_model`-dimensional embedding into a new `d_model`-dimensional space.
2. We then **reshape** that projection into `num_heads` groups of `d_k` dimensions each (e.g. 128 → 8 × 16).

So every head's slice comes *after* a learned projection that already mixed information from the entire embedding. Each head therefore draws on the whole word, just through a different learned "lens" — not from a fixed raw segment.

### Why do we need it? (Specialization)
Language is incredibly nuanced. If we only had one Single Head, the gradients would force the model to compromise and try to find a "middle ground" of attention. MHA lets the network look at a single word from 8 (or 16, or 32) different, specialized conceptual angles at once without that compromise.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # Dimension per head (e.g., 128 / 8 = 16)

        # bias=False, same reasoning as the single-head version.
        # Each of these projects the FULL d_model embedding -> d_model (not a slice).
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False) # The "Glue" at the end

    def forward(self, x, mask=None):
        batch_size, seq_len, d_model = x.size()

        # 1. Linear Projections (each looks at the WHOLE embedding)
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        # 2. RESHAPE the projected vectors into heads (we are NOT slicing the raw
        #    embedding -- W_q already mixed the whole vector; .view just regroups it).
        # (Batch, Seq_Len, d_model) -> (Batch, Seq_Len, Num_Heads, Head_Dim) -> (Batch, Num_Heads, Seq_Len, Head_Dim)
        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

        # 3. Apply Attention (Parallel over all heads!)
        output, _ = scaled_dot_product_attention(Q, K, V, mask=mask)

        # 4. Glue the Heads back together
        # Transpose back: (Batch, Num_Heads, Seq_Len, Head_Dim) -> (Batch, Seq_Len, Num_Heads, Head_Dim)
        # Contiguous/View: smashes Num_Heads and Head_Dim back into d_model
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)

        # 5. Final Output Projection
        return self.W_o(output)

# Look at the Dimensionality Tracking!
mha = MultiHeadAttention(d_model=128, num_heads=8)
final_output = mha(dummy_sentence_embeddings)

print(f"Input Shape : {dummy_sentence_embeddings.shape}")
print(f"Output Shape: {final_output.shape}")

## 4. Causal Masking (No Peeking at the Future)

So far every word could look at every other word — including words that come *later* in the sentence. That is fine for tasks that read a whole sentence at once (like classification). But an LLM is **autoregressive**: it generates text one token at a time, predicting the next word from the words so far. During training we feed the whole sentence at once for speed, so we must stop each position from "cheating" by looking at the answer (the words that come after it).

The fix is a **causal mask**: a lower-triangular matrix of 1s. Position `i` is only allowed to attend to positions `0..i` (itself and everything before it). Wherever the mask is 0 (the upper triangle = the future), our `scaled_dot_product_attention` sets the score to a huge negative number, so after softmax those future positions get ~zero weight.

Let's watch the upper triangle of the attention matrix collapse to 0.

In [ ]:
# Build a causal (lower-triangular) mask for a length-5 sequence.
# 1 = "allowed to look here", 0 = "blocked (it's in the future)".
seq_len = 5
causal_mask = torch.tril(torch.ones(seq_len, seq_len))
print("Causal mask (1 = allowed, 0 = blocked future):")
print(causal_mask.int())

# Reuse the hand-crafted "bank/river" embeddings from earlier, now WITH the mask.
_, masked_attn = scaled_dot_product_attention(emb, emb, emb, mask=causal_mask)
masked_attn = masked_attn[0]

print("\nMasked attention weights (upper triangle should be 0):\n")
header = "          " + "".join(f"{w:>8}" for w in words)
print(header)
for i, w in enumerate(words):
    row = "".join(f"{masked_attn[i, j]:8.2f}" for j in range(len(words)))
    print(f"{w:>10}{row}")

# The very first word can only attend to itself; each later row opens up one more column.
print("\nUpper-triangle (future) total weight, should be ~0:",
      round(masked_attn.triu(diagonal=1).sum().item(), 6))
print("Every row still sums to 1.0:", [round(v, 3) for v in masked_attn.sum(-1).tolist()])

plt.figure(figsize=(5, 4))
plt.imshow(masked_attn.numpy(), cmap="viridis")
plt.xticks(range(len(words)), words)
plt.yticks(range(len(words)), words)
plt.xlabel("Looking AT")
plt.ylabel("Querying word")
plt.title("Causal (masked) attention weights")
plt.colorbar()
plt.show()

### 🏋️ Try it yourself

1. **Read an attention matrix.** Run `MultiHeadAttention` on the hand-crafted `emb` (it's 4-D, so use `MultiHeadAttention(d_model=4, num_heads=2)`), but modify the class to also *return* the attention weights from one head, then print them. Which words does head 0 attend to?
2. **Apply a causal mask yourself.** Pass `mask=causal_mask` into `mha(...)` and confirm the model now refuses to let early words see later ones. (The starter code below already wires the mask through — just inspect the result.)

In [ ]:
# Task 2 starter: a multi-head layer that accepts a causal mask.
# (Our MultiHeadAttention already takes a `mask` argument and threads it through.)
mha_small = MultiHeadAttention(d_model=4, num_heads=2)
masked_out = mha_small(emb, mask=causal_mask)   # emb and causal_mask are from earlier cells
print("Output shape with causal mask:", masked_out.shape)

# Task 1 hint: to SEE the weights, make scaled_dot_product_attention's second
# return value escape the layer. For example, temporarily edit forward() to
# `out, w = scaled_dot_product_attention(Q, K, V, mask=mask); self.last_weights = w`
# then read `mha_small.last_weights[0, 0]` (batch 0, head 0) and print it.